# Sionna RT channel

> A channel ray-traced beforehand with Sionna RT, loaded from numpy arrays.

In [ ]:
#| default_exp sionna_rt

In [ ]:
#| hide
from nbdev.showdoc import *

A channel ray-traced beforehand with [Sionna RT](https://nvlabs.github.io/sionna/rt/), at the system level: a
`SystemLevelChannel` (see `system`) whose fading is the ray-traced table, a `RayTracedFading` (see
`propagation`). Each transmission gets the SNR of its nodes' positions,
`tx_power_w (the link's) × path gain / noise_power_w`, and succeeds with the packet error rate of that SNR.

The table, `channel.fading`, also works at the link level, which simulates every bit:
`LinkLevelChannel(fading=channel.fading, ...)` (see `link` and the example below).

Load the arrays with `SionnaRTChannel.from_files`. The coefficients must keep the path loss, so compute them with
`paths.cfr(..., normalize=False)` (see `RayTracedFading`).

In [ ]:
#| export
from __future__ import annotations

import math
from typing import Mapping, Sequence

import numpy as np

from comm_core.core import NodeId
from comm_core.propagation import RayTracedFading, RayTracedPathsFading, grid_points
from comm_core.system import FixedPower, SystemLevelChannel, bpsk_bit_error_rate, packet_error_rate

In [ ]:
#| export
class SionnaRTChannel(SystemLevelChannel):
    "A channel ray-traced beforehand (Sionna RT) between a set of positions, at the system level."

    def __init__(
        self,
        H, # Channel coefficients `H[rx, tx]` between the positions: (N, N) or (N, N, num_subcarriers), complex
        positions: Sequence, # The N positions (e.g. grid cells (x, y)), in the order of H's indices
        noise_power_w: float = 1e-13, # Noise power at the receiver, W (1e-13 W = -100 dBm)
        per_model: str = 'bpsk', # Packet error rate of the SNR: 'bpsk' or 'threshold' (see `packet_error_rate`)
        snr_threshold_db: float = 10.0, # The 'threshold' model's SNR
        node_positions: Mapping | None = None, # Where the nodes are to begin with (see `set_positions`)
        seed: int | None = None, # Seed of the packet losses
        power_control=None, # Transmit power and outages; None: the link's tx_power_w
    ):
        super().__init__(RayTracedFading(H, positions, node_positions), power_control or FixedPower(),
                         noise_power_w, per_model, snr_threshold_db, seed)

    @classmethod
    def from_files(
        cls,
        channel_path: str, # .npy of H[rx, tx]
        positions_path: str, # .npy of the positions
        **kwargs, # The other arguments of `SionnaRTChannel`
    ) -> SionnaRTChannel:
        "Load the arrays saved after ray tracing."
        return cls(np.load(channel_path), np.load(positions_path), **kwargs)

    def path_gain(self, sender: NodeId, receiver: NodeId) -> float:
        "Linear path gain |H|^2 from `sender` to `receiver` at their current positions (inf if co-located)."
        return self.fading.path_gain(sender, receiver)

    def snr_db(self, sender: NodeId, receiver: NodeId, tx_power_w: float = 1.0) -> float:
        "SNR of a transmission from `sender` to `receiver` with `tx_power_w` (dB)."
        snr = tx_power_w * self.path_gain(sender, receiver) / self.noise_power
        return 10 * math.log10(snr) if snr > 0 else -math.inf

In [ ]:
show_doc(SionnaRTChannel.from_files)

---

[source](https://github.com/Ahmed-Khaled-Saleh/comm-core/blob/main/comm_core/sionna_rt.py#L41){target="_blank" style="float:right; font-size:smaller"}

### SionnaRTChannel.from_files

```python
def from_files(
    channel_path:str, # .npy of H[rx, tx]
    positions_path:str, # .npy of the positions
    **kwargs
)->SionnaRTChannel: # The other arguments of `SionnaRTChannel`
```

*Load the arrays saved after ray tracing.*

In [ ]:
show_doc(SionnaRTChannel.set_positions)

---

[source](https://github.com/Ahmed-Khaled-Saleh/comm-core/blob/main/comm_core/system.py#LNone){target="_blank" style="float:right; font-size:smaller"}

### RadioChannel.set_positions

```python
def set_positions(
    positions
)->None:
```

*Where the nodes are, for a fading that depends on it (e.g. `RayTracedFading`).*

In [ ]:
show_doc(SionnaRTChannel.path_gain)

---

[source](https://github.com/Ahmed-Khaled-Saleh/comm-core/blob/main/comm_core/sionna_rt.py#L50){target="_blank" style="float:right; font-size:smaller"}

### SionnaRTChannel.path_gain

```python
def path_gain(
    sender:NodeId, receiver:NodeId
)->float:
```

*Linear path gain |H|^2 from `sender` to `receiver` at their current positions (inf if co-located).*

In [ ]:
show_doc(SionnaRTChannel.snr_db)

---

[source](https://github.com/Ahmed-Khaled-Saleh/comm-core/blob/main/comm_core/sionna_rt.py#L54){target="_blank" style="float:right; font-size:smaller"}

### SionnaRTChannel.snr_db

```python
def snr_db(
    sender:NodeId, receiver:NodeId, tx_power_w:float=1.0
)->float:
```

*SNR of a transmission from `sender` to `receiver` with `tx_power_w` (dB).*

### Tests

In [ ]:
from comm_core.core import Message, Transmission
from comm_core.topology import Link
import tempfile, os

In [ ]:
# 3 positions on a line: (0, 0) - (1, 0) - (5, 0); path gain falls with the distance; H[rx, tx]
positions = [(0, 0), (1, 0), (5, 0)]
gain_db = np.array([[0, -60, -95], [-60, 0, -90], [-95, -90, 0]], float)
H = np.sqrt(10 ** (gain_db / 10)) * np.exp(1j * np.arange(9).reshape(3, 3))
np.fill_diagonal(H, 0)
ch = SionnaRTChannel(H, positions, noise_power_w=1e-13, node_positions={'a': (0, 0), 'b': (1, 0), 'c': (5, 0)}, seed=0)
assert abs(ch.snr_db('a', 'b') - 70) < 1e-9 and abs(ch.snr_db('a', 'c') - 35) < 1e-9

def send(ch, sender, receiver, bits=256, n=200):
    t = Transmission(message=Message(sender, receiver, payload=1, size_bits=bits), sender=sender, receiver=receiver, start_time=0.0)
    results = [ch.transmit(t, Link(sender, receiver, tx_power_w=1e-4)) for _ in range(n)]    # 0.1 mW
    return sum(r.success for r in results) / n, results[0]

near, r = send(ch, 'a', 'b')                               # 0.1 mW x -60 dB / -100 dBm: 30 dB, always received
far, _ = send(ch, 'a', 'c')                                # -5 dB: lost
assert near == 1.0 and far == 0.0 and abs(r.snr_db - 30) < 1e-9 and abs(r.metadata['channel_gain_db'] + 60) < 1e-9
# the nodes move: the channel follows
ch.set_positions({'c': (1, 0)})
assert send(ch, 'a', 'c')[0] == 1.0
ch.set_positions({'b': (0, 0)})
assert ch.path_gain('a', 'b') == math.inf and send(ch, 'a', 'b')[0] == 1.0        # same cell
# in between: losses, reproducible with the seed
ch = SionnaRTChannel(H, positions, noise_power_w=2e-11, node_positions={'a': (0, 0), 'b': (1, 0)})   # 7 dB
ch.reset(seed=3); first = [send(ch, 'a', 'b', bits=1000, n=50)[0] for _ in range(3)]
ch.reset(seed=3); assert [send(ch, 'a', 'b', bits=1000, n=50)[0] for _ in range(3)] == first
assert 0 < first[0] < 1
# the threshold model, errors, files
ch = SionnaRTChannel(H, positions, per_model='threshold', snr_threshold_db=-10, node_positions={'a': (0, 0), 'c': (5, 0)})
assert send(ch, 'a', 'c')[0] == 1.0                                            # -5 dB, above the threshold
ch.set_positions({'c': (9, 9)})
try:
    send(ch, 'a', 'c'); raise AssertionError('no error')
except KeyError as e:
    assert 'no channel was ray-traced' in str(e)
with tempfile.TemporaryDirectory() as tmp:
    np.save(os.path.join(tmp, 'H.npy'), H); np.save(os.path.join(tmp, 'pos.npy'), np.array(positions))
    ch = SionnaRTChannel.from_files(os.path.join(tmp, 'H.npy'), os.path.join(tmp, 'pos.npy'))
    assert ch.fading.positions == [(0, 0), (1, 0), (5, 0)]

In a `Network`, the application moves the nodes before each step:

In [ ]:
from comm_core import Network, Node, FullMeshTopology, IdealProtocol, AnalyticalBackend

In [ ]:
ch = SionnaRTChannel(H, positions, noise_power_w=1e-8, seed=0)   # 1 W links: 20 dB at (1, 0), -15 dB at (5, 0)
net = Network([Node('a'), Node('b')], FullMeshTopology(['a', 'b']), IdealProtocol(), AnalyticalBackend(ch))
received = []
for pos_b in [(1, 0), (5, 0), (1, 0)]:                  # b walks away and comes back
    ch.set_positions({'a': (0, 0), 'b': pos_b})
    net.send(Message('a', 'b', payload='hi', size_bits=1024))
    net.step()
    received.append(len(net.receive('b')))
assert received == [1, 0, 1]

The same table at the link level. Every bit is simulated, and channel inversion uses just the power each link
needs:

In [ ]:
from comm_core.link import LinkLevelChannel
from comm_core.system import ChannelInversion

In [ ]:
table = SionnaRTChannel(H, positions).fading
ch = LinkLevelChannel(fading=table, power_control=ChannelInversion(target_snr_db=10, max_power=1.0), noise_power=1e-13, seed=0)
nodes = ['a', 'b', 'c']
net = Network([Node(n) for n in nodes], FullMeshTopology(nodes), IdealProtocol(), AnalyticalBackend(ch))
ch.set_positions({'a': (0, 0), 'b': (1, 0), 'c': (5, 0)})      # passed on to the table
for s in nodes:
    for r in nodes:
        if s != r:
            net.send(Message(s, r, payload=np.arange(8)))
results = {(x.transmission.sender, x.transmission.receiver): x for x in net.step()}
# 10 dB over -100 dBm: a-b (-60 dB) needs 1e-6 W, b-c (-90 dB) 1e-3 W, a-c (-95 dB) 3.2e-3 W, all within 1 W
assert all(x.success for x in results.values())
assert abs(results['a', 'b'].metadata['tx_power'] - 1e-6) < 1e-12 and abs(results['a', 'c'].snr_db - 10) < 1e-6
ch.power_control.max_power = 2e-3                              # 2 mW: a-c is not used
results = {(s, r): ch.transmit(Transmission(Message(s, r, np.arange(8)), s, r, 0.0), Link(s, r)).success
           for s in nodes for r in nodes if s != r}
assert results == {('a', 'b'): True, ('b', 'a'): True, ('b', 'c'): True, ('c', 'b'): True, ('a', 'c'): False, ('c', 'a'): False}

## Continuous positions: paths at anchors

`SionnaRTChannel` and `RayTracedFading` take a channel at fixed points, which suits discrete environments. For
continuous ones, `trace_paths` ray-traces the *paths* between anchor points in a Sionna RT scene, e.g. every 5 cm
(`grid_points`). `RayTracedPathsFading` (see `propagation`) then moves them to wherever the nodes are.

Trace once, where Sionna RT runs (e.g. a GPU node), and save:

```python
from sionna.rt import load_scene
scene = load_scene('my_room.xml')
scene.frequency = 5.18e9
access_point = [[0.0, 0.0, 2.4]]
robot_area = grid_points((-6.5, -3.0), (-3.0, -0.5), 0.05, height=0.4)      # every 5 cm, antennas at 0.4 m
paths = trace_paths(scene, access_point, robot_area, max_depth=6)
RayTracedPathsFading(**paths).save('room_paths.npz')
```

Then use the saved paths anywhere, without Sionna:

```python
fading = RayTracedPathsFading.from_npz('room_paths.npz', max_distance=0.1)
channel = SystemLevelChannel(fading=fading, noise_power=..., ...)
channel.set_positions({'ap': (0, 0, 2.4), 'robot': (-4.213, -1.871, 0.4)})   # anywhere in the area
```

**Agent-to-agent links** use the same points on both sides: `trace_paths(scene, points, points)`.

**What it costs.** The table holds every transmitter anchor × receiver anchor × path:

* **Access point to robot:** one transmitter, so anchors every 5 cm are cheap. A 10 × 10 m area is 40k anchors,
  about 100 MB with 30 paths.
* **Agent to agent:** the number of pairs grows with the square of the number of anchors, so anchors every 5 cm
  are only possible over small areas. Moving the paths, rather than taking the nearest anchor's channel, is what
  lets the anchors be further apart: see the validation below.

Coincident transmitter and receiver anchors get no paths, so nodes that share an anchor have a perfect link (see
`RayTracedPathsFading`).

In [ ]:
#| export
def _unit(theta, phi):
    "Unit vectors of zenith / azimuth angles."
    return np.stack([np.sin(theta) * np.cos(phi), np.sin(theta) * np.sin(phi), np.cos(theta)], -1)


def trace_paths(
    scene, # A Sionna RT scene with its frequency set; its tx_array / rx_array, or one isotropic antenna each
    tx_points, # Transmitter anchors, (N_tx, 3)
    rx_points, # Receiver anchors, (N_rx, 3)
    batch: int = 8, # Transmitters, and receivers, per PathSolver call (more overflow its path buffer and drop paths)
    progress: bool = False, # Print the progress
    **solver_kwargs, # PathSolver options; default max_depth=3, line of sight, specular reflections, refraction
) -> dict:
    "Ray-trace the paths between every transmitter and receiver anchor: the arguments of `RayTracedPathsFading`."
    import mitsuba as mi
    from sionna.rt import PathSolver, PlanarArray, Receiver, Transmitter
    tx_points = np.asarray(tx_points, np.float64).reshape(-1, 3)
    rx_points = np.asarray(rx_points, np.float64).reshape(-1, 3)
    if scene.tx_array is None:
        scene.tx_array = PlanarArray(num_rows=1, num_cols=1, pattern='iso', polarization='V')
    if scene.rx_array is None:
        scene.rx_array = PlanarArray(num_rows=1, num_cols=1, pattern='iso', polarization='V')
    kwargs = dict(max_depth=3, los=True, specular_reflection=True, diffuse_reflection=False, refraction=True,
                  synthetic_array=True) | solver_kwargs
    solver = PathSolver()
    own = [*scene.transmitters.values(), *scene.receivers.values()]     # the scene's radios, put back afterwards
    for radio in own:
        scene.remove(radio.name)
    point = lambda p: mi.Point3f(*map(float, p))
    blocks, num_paths = [], 0
    calls = -(-len(tx_points) // batch) * -(-len(rx_points) // batch)
    try:
        for i0 in range(0, len(tx_points), batch):
            for j0 in range(0, len(rx_points), batch):
                tx, rx = tx_points[i0:i0 + batch], rx_points[j0:j0 + batch]
                for k, p in enumerate(tx):
                    scene.add(Transmitter(f'cc-tx-{k}', position=point(p)))
                for k, p in enumerate(rx):
                    scene.add(Receiver(f'cc-rx-{k}', position=point(p)))
                paths = solver(scene, **kwargs)
                a, tau = paths.cir(normalize_delays=False, out_type='numpy')
                a = a[:, 0, :, 0, :, 0]                                   # (rx, tx, paths): one antenna each
                valid = paths.valid.numpy().astype(bool) & (np.abs(a) > 0) & np.isfinite(a) & np.isfinite(tau)
                valid &= ~(np.abs(rx[:, None, None] - tx[None, :, None]).sum(-1) < 1e-9)   # coincident anchors
                u_tx = _unit(paths.theta_t.numpy(), paths.phi_t.numpy())
                u_rx = _unit(paths.theta_r.numpy(), paths.phi_r.numpy())
                sw = lambda x: np.swapaxes(x, 0, 1)                       # (tx, rx, ...)
                blocks.append((i0, j0, sw(np.where(valid, a, 0)), sw(np.where(valid, tau, 0)),
                               sw(np.where(valid[..., None], u_tx, 0)), sw(np.where(valid[..., None], u_rx, 0))))
                num_paths = max(num_paths, a.shape[-1])
                for k in range(len(tx)):
                    scene.remove(f'cc-tx-{k}')
                for k in range(len(rx)):
                    scene.remove(f'cc-rx-{k}')
                if progress:
                    print(f'{len(blocks)} / {calls}', end='\r')
    finally:
        for name in [n for n in (*scene.transmitters, *scene.receivers) if n.startswith('cc-')]:
            scene.remove(name)
        for radio in own:
            scene.add(radio)
    shape = (len(tx_points), len(rx_points), num_paths)
    out = dict(a=np.zeros(shape, np.complex128), tau=np.zeros(shape), u_tx=np.zeros(shape + (3,)), u_rx=np.zeros(shape + (3,)))
    for i0, j0, a, tau, u_tx, u_rx in blocks:
        at = (slice(i0, i0 + a.shape[0]), slice(j0, j0 + a.shape[1]), slice(0, a.shape[-1]))
        out['a'][at], out['tau'][at], out['u_tx'][at], out['u_rx'][at] = a, tau, u_tx, u_rx
    frequency = float(np.asarray(scene.frequency).reshape(-1)[0])
    return dict(tx_points=tx_points, rx_points=rx_points, **out, frequency=frequency)

In [ ]:
show_doc(trace_paths)

---

[source](https://github.com/Ahmed-Khaled-Saleh/comm-core/blob/main/comm_core/sionna_rt.py#L65){target="_blank" style="float:right; font-size:smaller"}

### trace_paths

```python
def trace_paths(
    scene, # A Sionna RT scene with its frequency set; its tx_array / rx_array, or one isotropic antenna each
    tx_points, # Transmitter anchors, (N_tx, 3)
    rx_points, # Receiver anchors, (N_rx, 3)
    batch:int=8, # Transmitters, and receivers, per PathSolver call (more overflow its path buffer and drop paths)
    progress:bool=False, # Print the progress
    **solver_kwargs
)->dict: # PathSolver options; default max_depth=3, line of sight, specular reflections, refraction
```

*Ray-trace the paths between every transmitter and receiver anchor: the arguments of `RayTracedPathsFading`.*

### Validation against Sionna RT

The test: Sionna's street-canyon scene at 5.18 GHz, with reflections off the buildings and the ground (up to 3
bounces). The channel at 40 random positions between the anchors, moved from the anchors, is compared with the
channel ray-traced directly at those positions, over 52 subcarriers (20 MHz). For reference, two other ways are
shown: plane waves (`Δτ = −(u_tx·Δ_tx + u_rx·Δ_rx)/c`, as in a first-order extrapolation) and the nearest
anchor's channel. Lower is better; −20 dB means errors of 1% of the channel's power.

This needs Sionna RT, so it does not run in the tests; the output was saved when it ran (Sionna 2.2 with Sionna RT 2.2, on a CPU).

In [ ]:
#| eval: false
import sionna.rt as rt
from sionna.rt import load_scene

scene = load_scene(rt.scene.simple_street_canyon)
scene.frequency = 5.18e9
subcarriers = (np.arange(52) - 25.5) * 312.5e3                      # 20 MHz, 52 subcarriers
rng = np.random.default_rng(0)

def response(a, tau):
    return np.exp(-2j * np.pi * np.outer(subcarriers, tau)) @ a

def plane_waves(fading, s, r):
    "For comparison, the first-order move: Δτ = −(u_tx·Δ_tx + u_rx·Δ_rx) / c."
    a, tau, u_dep, d_dep, u_arr, d_arr = fading._paths(s, r)
    live = a != 0
    dtau = -(u_dep[live] @ d_dep + u_arr[live] @ d_arr) / 299_792_458.0
    return response(a[live] * np.exp(-2j * np.pi * fading.frequency * dtau), tau[live] + dtau)

def compare(label, tx_anchors, rx_anchors, pairs):
    fading = RayTracedPathsFading(**trace_paths(scene, tx_anchors, rx_anchors), subcarriers=subcarriers)
    errors = {'moved paths': [], 'plane waves': [], 'nearest anchor': []}
    power, same_paths = [], []
    for pt, pr in pairs:
        direct = trace_paths(scene, pt[None], pr[None])                # the truth: traced right there
        truth = response(direct['a'][0, 0], direct['tau'][0, 0])
        fading.set_positions({'s': pt, 'r': pr})
        i = np.argmin(((fading.tx_points - pt) ** 2).sum(-1))
        j = np.argmin(((fading.rx_points - pr) ** 2).sum(-1))
        for name, estimate in (('moved paths', fading.cfr('s', 'r')), ('plane waves', plane_waves(fading, 's', 'r')),
                               ('nearest anchor', response(fading.a[i, j], fading.tau[i, j]))):
            errors[name].append(np.sum(np.abs(estimate - truth) ** 2))
        power.append(np.sum(np.abs(truth) ** 2))
        same_paths.append((fading.a[i, j] != 0).sum() == (direct['a'][0, 0] != 0).sum())
    nmse = lambda e, keep: 10 * np.log10(np.sum(np.array(e)[keep]) / np.sum(np.array(power)[keep]))
    everything, same = np.ones(len(pairs), bool), np.array(same_paths)
    print(f'{label}: ' + ', '.join(f'{name} {nmse(e, everything):.1f} dB' for name, e in errors.items()))
    if not same.all():
        print(f'    {(~same).sum()} of {len(pairs)} positions have other paths than their anchors; '
              f'without them, moved paths {nmse(errors["moved paths"], same):.1f} dB')

access_point = np.array([[-30.0, 0.0, 6.0]])
lo, hi = (10.0, -1.0), (11.0, 0.0)
robots = [(access_point[0], np.append(rng.uniform(lo, hi), 1.5)) for _ in range(40)]
for spacing in (0.05, 0.2, 0.5):
    compare(f'access point -> robot, anchors every {spacing * 100:.0f} cm', access_point,
            grid_points(lo, hi, spacing, height=1.5), robots)
area_a, area_b = ((-5.0, -1.0), (-4.0, 0.0)), ((5.0, -1.0), (6.0, 0.0))
agents = [(np.append(rng.uniform(*area_a), 1.5), np.append(rng.uniform(*area_b), 1.5)) for _ in range(30)]
compare('agent -> agent, anchors every 25 cm', grid_points(*area_a, 0.25, height=1.5),
        grid_points(*area_b, 0.25, height=1.5), agents)

access point -> robot, anchors every 5 cm: moved paths -62.3 dB, plane waves -60.5 dB, nearest anchor 2.0 dB
access point -> robot, anchors every 20 cm: moved paths -56.8 dB, plane waves -44.2 dB, nearest anchor 2.4 dB
access point -> robot, anchors every 50 cm: moved paths -47.5 dB, plane waves -29.4 dB, nearest anchor 2.9 dB
agent -> agent, anchors every 25 cm: moved paths -29.1 dB, plane waves -20.9 dB, nearest anchor 2.4 dB
    4 of 30 positions have other paths than their anchors; without them, moved paths -43.5 dB


* **A fixed access point:** moving the paths keeps the channel within −47 to −63 dB of the one traced directly,
  even with anchors 50 cm apart.
* **Plane waves** need denser anchors for the same accuracy.
* **The nearest anchor's channel** is unusable at any spacing: at a few centimetres from the anchor, its phase is
  already unrelated to the true one.
* **Both ends moving:** at some positions a reflection appears or disappears, e.g. at a building's edge, so the
  paths there differ from the anchors' paths. Those positions set the average error. Elsewhere it is −43.5 dB, and
  denser anchors only help near such edges.

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()